# 🗂️ Subset & Model Selection
**ISLP Chapter 6 · Pattern Recognition for the Rest of Us**

> When you have many predictors, which ones should you include in the model? Subset selection methods systematically compare model subsets. Information criteria (Cp, BIC, adjusted R²) give principled ways to choose model size without a test set.

### What you'll learn
- Best subset selection: exhaustive search
- Forward and backward stepwise selection: greedy approximations
- Cp, BIC, and adjusted R² as model selection criteria
- Practical comparison with cross-validation

### Dataset: Hitters (ISLP) — 19 baseball predictors
### Time: ~50 minutes

## 🎯 Part 1 — Why Variable Selection?

With p predictors, using all of them can overfit — especially when p is large relative to n. Including irrelevant predictors:
- Increases variance of estimates
- Reduces interpretability
- May worsen prediction on new data

We want the smallest model that explains the data well.

In [ ]:
# Forward stepwise selection (greedy — each step adds best remaining feature)
def forward_stepwise(X, y, feature_names, max_features=None):
    if max_features is None: max_features = X.shape[1]
    selected = []
    remaining = list(range(X.shape[1]))
    results = []
    lr = LinearRegression()
    for step in range(max_features):
        best_score, best_feat = -np.inf, None
        for feat in remaining:
            candidate = selected + [feat]
            cv = cross_val_score(lr, X[:, candidate], y, cv=5,
                                scoring='neg_mean_squared_error').mean()
            if cv > best_score:
                best_score, best_feat = cv, feat
        selected.append(best_feat)
        remaining.remove(best_feat)
        results.append({'n_features': step+1, 'features': selected.copy(),
                       'cv_mse': -best_score,
                       'last_added': feature_names[best_feat]})
    return pd.DataFrame(results)

print('Running forward stepwise selection...')
fwd_results = forward_stepwise(X, y, features, max_features=15)
print(fwd_results[['n_features','last_added','cv_mse']].to_string())
print(f'\nBest model size: {fwd_results["cv_mse"].idxmin()+1} features')

In [ ]:
# Plot: CV MSE vs number of features
fig, ax = plt.subplots(figsize=(10, 4))
best_n = fwd_results['cv_mse'].idxmin()
ax.plot(fwd_results['n_features'], fwd_results['cv_mse'], 'o-', color='#1e5fa8', lw=2, markersize=6)
ax.axvline(fwd_results.loc[best_n, 'n_features'], color='#e85d20', lw=1.5, ls='--',
          label=f'Best: {fwd_results.loc[best_n,"n_features"]} features (CV MSE={fwd_results.loc[best_n,"cv_mse"]:.4f})')
ax.set_xlabel('Number of features')
ax.set_ylabel('5-fold CV MSE (log salary)')
ax.set_title('Forward Stepwise Selection — Hitters Dataset')
ax.legend()
plt.tight_layout(); plt.show()
best_features = fwd_results.loc[best_n, 'features']
print(f'\nSelected features: {[features[i] for i in best_features]}')

In [ ]:
# Information criteria: AIC, BIC, adjusted R²
# (computed on full training data — no cross-validation needed)
import statsmodels.api as sm

n = len(y)
X_sm = sm.add_constant(X[:, :10])  # first 10 features for illustration
model_sm = sm.OLS(y, X_sm).fit()

print('Model selection criteria (statsmodels):')  
print(f'  AIC:  {model_sm.aic:.2f}')
print(f'  BIC:  {model_sm.bic:.2f}')
print(f'  R²:   {model_sm.rsquared:.4f}')
print(f'  Adj R²: {model_sm.rsquared_adj:.4f}')
print()
print('Building models of increasing size and comparing criteria...')
for k in [1, 3, 5, 8, 10, 12, 15]:
    if k <= len(features):
        Xk = sm.add_constant(X[:, :k])
        mk = sm.OLS(y, Xk).fit()
        print(f'  p={k:<3} AIC={mk.aic:.1f}  BIC={mk.bic:.1f}  AdjR²={mk.rsquared_adj:.4f}')

In [ ]:
# Exercise workspace
# Task 1: Implement backward stepwise selection (start with all features, remove worst one each step)
# YOUR CODE HERE

# Task 2: Compare forward stepwise vs Lasso (from regularization notebook) on Hitters
# Which selects fewer features? Which has lower CV MSE?
from sklearn.linear_model import LassoCV
# YOUR CODE HERE

# Task 3: What features does the best 5-feature model select? Interpret each one
best_5 = fwd_results.loc[fwd_results['n_features']==5, 'features'].values[0]
print('Top 5 features:', [features[i] for i in best_5])
# YOUR CODE HERE — fit and report coefficients

In [ ]:
answers = {
    'q1': '',  # What is best subset selection and why is it computationally expensive?
    'q2': '',  # What is the difference between forward and backward stepwise selection?
    'q3': '',  # What does BIC penalize compared to AIC?
    'q4': '',  # Why is adjusted R² better than R² for model comparison?
    'q5': '',  # When would you use stepwise selection vs Lasso for variable selection?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print('❌ Still empty: '+str(missing) if missing else '✅ Done! File → Save a copy in GitHub')

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Subset & Model Selection"
# ══════════════════════════════════════════════════════════════════════
# 🤖 AI-GRADED SUBMISSION
# ══════════════════════════════════════════════════════════════════════
#
# REQUIRED — fill in your GitHub username so your instructor can
# match your submission to your name:
#
GITHUB_USERNAME = ""   # ← e.g. "jsmith42"  — your github.com username
#
# ── ONE-TIME SETUP (do this once, works for all 30 notebooks) ─────────
#
# You need a FREE Gemini API key from Google AI Studio.
#
# ⚠️  IMPORTANT: Use your PERSONAL Gmail — NOT your university email.
#     Many universities block Google AI Studio on managed accounts.
#     A personal @gmail.com works everywhere and is always free.
#
# Steps:
#   1. Open a private/incognito browser window
#   2. Go to: aistudio.google.com
#   3. Sign in with your PERSONAL Gmail (@gmail.com)
#   4. Click "Get API key" → "Create API key" → copy it
#   5. Back in Colab: click the 🔑 icon in the LEFT SIDEBAR
#      → "+ Add new secret"
#      → Name:  GEMINI_API_KEY
#      → Value: paste your key here
#      → Toggle the switch ON
#   6. Re-run this cell
#
# Your key is saved to your Colab account — works across all notebooks.
# It is FREE — no credit card, no billing required.
#
# ── NO KEY? ────────────────────────────────────────────────────────────
# You still get completion-based feedback without a key.
# You just won't get AI analysis of your specific answers.
#
# ══════════════════════════════════════════════════════════════════════
# DO NOT EDIT BELOW THIS LINE
# ══════════════════════════════════════════════════════════════════════
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Add more detail to show your understanding."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with reasonable detail. "
                             "Add a free Gemini key for AI analysis of your specific answers."),
                "tip": "Get a free key at aistudio.google.com — use your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             "Complete the remaining questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Set GITHUB_USERNAME above to track submissions")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
print("\n" + "\u2500"*58)
print("  Checking submission...")
print("\u2500"*58)

if "answers" not in globals():
    print("  \u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    if username:
        print(f"  Student  : @{username}")
    else:
        print(f"  Student  : \u26a0\ufe0f  GITHUB_USERNAME not set")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)")
        print()
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell")
        print()
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be Save: File \u2192 Save a copy in GitHub \u2192 your fork")
    print()
